###cricuits

In [0]:
class Silver_circuits():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('streaming_project.bronze.circuits')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)

        colrename_df= (colrename_df.withColumnRenamed('lat','latitude')
                            .withColumnRenamed('lng','longitude')
                            .withColumnRenamed('alt','altitude')
                            .withColumnRenamed('name','circuit_name')
                            .withColumnRenamed('country','circuit_country')
                            .withColumnRenamed('location','circuit_location')
        )
        from pyspark.sql.functions import expr
        colrename_df=(colrename_df.withColumn('circuit_country',
                                        expr("case when circuit_country = 'USA' then 'United States'"
                                               "when circuit_country = 'UK' then 'United Kingdom'"
                                               "when circuit_country = 'UAE' then 'United Arab Emirates'"
                                               "else  circuit_country end as circuit_country")
                                            )
                       )
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col
        apply_tran_df= (colrename_df.selectExpr("circuit_id","circuit_ref","circuit_name",
                                              "circuit_location","circuit_country","latitude",
                                              "longitude","altitude","circuits_ingestion_date","source")
                        )
        return apply_tran_df
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("streaming_project.silver.circuits")
        print("Data write into sliver circuits table is Done")
        
    
       
        
    def process(self):
        print("Started silver-ingestion-circuits  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
       
    


###constructors

In [0]:
class Silver_constructors():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('streaming_project.bronze.constructors')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
            colrename_df= (colrename_df.withColumnRenamed('name','constructor_team')
                                     .withColumnRenamed('nationality','constructor_nationality')
                           )
            
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col
        apply_tran_df= (colrename_df.selectExpr("constructor_id","constructor_ref","constructor_team","constructor_nationality","constructors_ingestion_date","source")
                        )
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("streaming_project.silver.constructors")
        print("Data write into sliver constructors table is Done")
       
    
    def process(self):
        print("Started silver-ingestion-constructors  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
        
    


###drivers

In [0]:
class Silver_drivers():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('streaming_project.bronze.drivers')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
            colrename_df =(colrename_df.withColumnRenamed('name','driver_name')
                                      .withColumnRenamed('code','driver_code')
                                      .withColumnRenamed('number','driver_number')
                                     .withColumnRenamed('nationality','driver_nationality')
                          )
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when,concat,lit
        apply_tran_df=colrename_df.withColumn('driver_name',
                                              concat(col('driver_name.forename'),lit(" "),col('driver_name.surname')))
        apply_tran_df=apply_tran_df.withColumn('driver_code',when(col('driver_code').isin('\\N'),None).otherwise(col('driver_code')))
        apply_tran_df=apply_tran_df.select(col('driver_id'),col('driver_ref'),col('driver_number'),col('driver_code'),
                                           col('driver_name'),col('dob'),col('driver_nationality'),col('drivers_ingestion_date'),col('source'))
        
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("streaming_project.silver.drivers")
        print("Data write into sliver drivers table is Done")
       

    
    def process(self):
        print("Started silver-ingestion-drivers  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
       
    

###lap_times

In [0]:
class Silver_lap_times():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        from pyspark.sql.functions import col,max,count
        if spark.catalog.tableExists("streaming_project.silver.lap_times"):
            max_ingestion_date=spark.read.table('streaming_project.silver.lap_times').agg(max(col('lap_times_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if max_ingestion_date is None:
                max_ingestion_date='1900-01-01 00:00:00'
            print("if_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.lap_times').filter(col('LapTimesIngestionDate')>max_ingestion_date)
    
            print("reading lap_times read_df")
            display(read_df.select(count("*")))
        else:
            max_ingestion_date='1900-01-01 00:00:00'
            print("else_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.lap_times').filter(col('LapTimesIngestionDate')>max_ingestion_date)
        
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col
        apply_tran_df = colrename_df.withColumnRenamed('time','time_minutes')
        apply_tran_df = apply_tran_df.select(col('race_id'),col('driver_id'),col('lap'),col('position'),col('time_minutes'),col('milliseconds'),col('lap_times_ingestion_date'),col('source'))
        
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.partitionBy("race_id").mode("append").saveAsTable("streaming_project.silver.lap_times")
        print("Data write into sliver lap_times table is Done")
    
    def process(self):
        print("Started silver-ingestion-lap_times  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
        
    

###pit_stops

In [0]:
class Silver_pit_stops():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        from pyspark.sql.functions import col,max,count
        if spark.catalog.tableExists("streaming_project.silver.pit_stops"):
            max_ingestion_date=spark.read.table('streaming_project.silver.pit_stops').agg(max(col('pit_stops_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if max_ingestion_date is None:
                max_ingestion_date='1900-01-01 00:00:00'
            print("if_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.pit_stops').filter(col('PitStopsIngestionDate')>max_ingestion_date)
    
            print("reading pit_stops read_df")
            display(read_df.select(count("*")))
        else:
            max_ingestion_date='1900-01-01 00:00:00'
            print("else_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.pit_stops').filter(col('PitStopsIngestionDate')>max_ingestion_date)
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col
        apply_tran_df= (colrename_df.withColumn('duration_sec',round(col('milliseconds')/1000,4))
                        .withColumn('duration_minutes',round(col('milliseconds')/60000,4))
                        .select(col('driver_id'),col('race_id'),col('stop'),col('lap'),col('time'),col('duration_sec'),col('duration_minutes'),col('milliseconds'),col('pit_stops_ingestion_date'),col('source'))
                        )
    
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.partitionBy("race_id").mode("append").saveAsTable("streaming_project.silver.pit_stops")
        print("Data write into sliver pit_stops table is Done")
     
    def process(self):
        print("Started silver-ingestion-pit_stops  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
        





###qualifying

In [0]:
class Silver_qualifying():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        from pyspark.sql.functions import col,max,count
        if spark.catalog.tableExists("streaming_project.silver.qualifying"):
            max_ingestion_date=spark.read.table('streaming_project.silver.qualifying').agg(max(col('qualifying_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if max_ingestion_date is None:
                max_ingestion_date='1900-01-01 00:00:00'
            print("if_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.qualifying').filter(col('QualifyingIngestionDate')>max_ingestion_date)
    
            print("reading qualifying read_df")
            display(read_df.select(count("*")))
        else:
            max_ingestion_date='1900-01-01 00:00:00'
            print("else_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.qualifying').filter(col('QualifyingIngestionDate')>max_ingestion_date)
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when
        apply_tran_df=colrename_df
        for c,t in apply_tran_df.dtypes:
            if t=='string':
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isin('\\N',''),'-').otherwise(col(c)))

        apply_tran_df= (apply_tran_df.select(col('race_id'),col('driver_id'),col('constructor_id'),
                                             col('number'),col('position'),col('q1'),col('q2'),col('q3'),
                                             col('qualifying_ingestion_date'),col('source'))
                        )
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.partitionBy("race_id").mode("append").saveAsTable("streaming_project.silver.qualifying")
        print("Data write into sliver qualifying table is Done")
    
     
    def process(self):
        print("Started silver-ingestion-qualifying  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
      





###results

In [0]:
class Silver_results():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        from pyspark.sql.functions import col,max,count
        if spark.catalog.tableExists("streaming_project.silver.results"):
            max_ingestion_date=spark.read.table('streaming_project.silver.results').agg(max(col('results_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if max_ingestion_date is None:
                max_ingestion_date='1900-01-01 00:00:00'
            print("if_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.results').filter(col('ResultsIngestionDate')>max_ingestion_date)
    
            print("reading results read_df")
            display(read_df.select(count(col('resultId'))))
        else:
            max_ingestion_date='1900-01-01 00:00:00'
            print("else_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('streaming_project.bronze.results').filter(col('ResultsIngestionDate')>max_ingestion_date)
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when
        apply_tran_df=colrename_df
        for c,t in apply_tran_df.dtypes:
            if t=='string':
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isin('\\N'),'-').otherwise(col(c)))
            elif t in ['int', 'bigint']:
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isNull(),0000).otherwise(col(c)))
            elif t in ['float', 'double']:
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isNull(),0.0).otherwise(col(c)))
                
        apply_tran_df= (apply_tran_df.selectExpr("result_id","race_id","driver_id","constructor_id",
                                                "number","grid","position as result_position","position_text as result_position_text","position_order as result_position_order",
                                                "points as result_points","laps","time",
                                                "milliseconds","fastest_lap","rank as fastest_lap_rank",
                                                "fastest_lap_time","fastest_lap_speed",
                                                "status_id","results_ingestion_date","source")
                        )
        
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.partitionBy("race_id").mode("append").saveAsTable("streaming_project.silver.results")
        print("Data write into sliver results table is Done")
        
       
     
    def process(self):
        print("Started silver-ingestion-results  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
       





###races

In [0]:
class Silver_races():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    bronze_path="streaming_project/bronze"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('streaming_project.bronze.races')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when,to_timestamp,lit,concat,try_to_timestamp
        apply_tran_df=colrename_df
        for c,t in apply_tran_df.dtypes:
            if t=='string':
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isin('\\N'),None).otherwise(col(c)))
        
        apply_tran_df = apply_tran_df.withColumn("timestamp",
                                            to_timestamp(
                                                concat(col("date") ,lit(" ") , col("time")),
                                                "yyyy-MM-dd HH:mm:ss"
                                            ))
        apply_tran_df= (apply_tran_df.selectExpr('race_id','year as race_year','round','circuit_id','name as race_name',
                                                 'date as race_date','time as race_time','timestamp','races_ingestion_date','source'
                        ))
        
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("streaming_project.silver.races")
        print("Data write into sliver races table is Done")
        
    
        
     
    def process(self):
        print("Started silver-ingestion-races  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
        


In [0]:

# from pyspark.sql.functions import col,round
# import re
# for c in df.columns:
#     result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
#     column_name="_".join(result)
#     print(column_name)

# df.printSchema()

In [0]:
# for c,t in df.dtypes:
#     if t=='string':
#         print(c)
#         df_null=df.filter(col(c).isin('\\N'))
#         display(df_null)
    